# Giai đoạn 4: Tiền xử lý tiếng Việt, Chống rò rỉ nhãn & Vector hóa dùng chung

Notebook này đọc file `data/processed/stories.csv` (**`20.852` mẫu** từ 4 nguồn, **`35` nhãn chuẩn**) và thực hiện:
1. **Làm sạch văn bản (`clean_text` trong `src/preprocessing/clean.py`):** Chuẩn hóa Unicode **NFC** (xử lý `1.036` văn án lỗi dấu tổ hợp `NFD`), xóa HTML/URL, **tách từ dính tại ranh giới hoa/thường (`split_case_boundaries` & `CAMELCASE_WHITELIST`)** *trước* khi đưa về chữ thường, xóa ký tự đặc biệt.
2. **Chống rò rỉ nhãn bằng Regex (`remove_label_leakage` trong `src/preprocessing/leakage.py`):** Cắt cụm khai báo mở đầu `"Thể loại: ... (Văn án|Giới thiệu|Tóm tắt|Nội dung):"` (`2.881 -> 0`) và xóa chính xác từ khóa chuyên biệt trong `LEAKAGE_KEYWORDS`.
3. **Tách từ & lọc Stopwords (`tokenize_vietnamese`):** Dùng `underthesea` và lọc **120 từ dừng** (`data/stopwords/vietnamese_stopwords.txt`).
4. **Chia tập phân tầng đa nhãn (`IterativeStratification`, 70/15/15, `RANDOM_STATE = 42`):** Chia `train = 14.613`, `val = 3.125`, `test = 3.114` vào **`data/processed/stories_clean.csv`** (`52` cột) **trước** khi fit `TfidfVectorizer` (`10.000` chiều, 1–2 gram) vào `models/tfidf_vectorizer.pkl`.

In [1]:
import re
import sys
import unicodedata
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import (
    GENRE_LABELS,
    LEAKAGE_KEYWORDS,
    STOPWORDS_PATH,
    STORIES_CLEAN_CSV,
    STORIES_CSV,
    TFIDF_VECTORIZER_PATH,
)
from src.pipeline import assign_splits
from src.preprocessing.clean import (
    CAMELCASE_WHITELIST,
    WORD_RE,
    clean_text,
    split_case_boundaries,
)
from src.preprocessing.leakage import (
    HEADER_GENRE_BLOCK_RE,
    LEADING_LABEL_PREFIX_RE,
    remove_label_leakage,
)
from src.preprocessing.tokenizer import load_stopwords, tokenize_vietnamese

df = pd.read_csv(STORIES_CSV, encoding="utf-8")
print(f"Đã đọc {len(df)} mẫu từ {STORIES_CSV}")
print("Các cột hiện có:", list(df.columns))

Đã đọc 20852 mẫu từ C:\Users\ACER\Downloads\NLP_CUỐI KỲ\data\processed\stories.csv
Các cột hiện có: ['id', 'title', 'author', 'summary', 'genres_raw', 'url', 'source', 'crawl_date', 'genres', 'n_labels', 'n_words', 'label_ngon_tinh', 'label_dam_my', 'label_bach_hop', 'label_tong_tai', 'label_nu_phu', 'label_nu_cuong', 'label_sung', 'label_nguoc', 'label_hai_huoc', 'label_tien_hiep', 'label_kiem_hiep', 'label_huyen_huyen', 'label_di_gioi', 'label_di_nang', 'label_mat_the', 'label_khoa_huyen', 'label_linh_di', 'label_trinh_tham', 'label_tham_hiem', 'label_vong_du', 'label_he_thong', 'label_xuyen_khong', 'label_trong_sinh', 'label_co_dai', 'label_hien_dai', 'label_hoc_duong', 'label_cung_dau', 'label_gia_dau', 'label_dien_van', 'label_lich_su', 'label_quan_su', 'label_quan_truong', 'label_doan_van', 'label_tieu_thuyet', 'label_khac', 'status', 'chapters', 'rating']


In [2]:
# Bước 1: Chuẩn hóa NFC, tách từ nối liền (split_case_boundaries) & làm sạch cơ bản (clean_text)
def _count_case_stuck(series: pd.Series) -> int:
    cnt = 0
    for text in series.fillna(""):
        s = unicodedata.normalize("NFC", str(text))
        found = False
        for m in WORD_RE.finditer(s):
            tok = m.group(0)
            if len(tok) < 3 or tok.lower() in CAMELCASE_WHITELIST:
                continue
            for i in range(1, len(tok)):
                if tok[i - 1].islower() and tok[i].isupper():
                    found = True
                    break
                if i + 1 < len(tok) and tok[i - 1].isupper() and tok[i].isupper() and tok[i + 1].islower():
                    found = True
                    break
            if found:
                break
        if found:
            cnt += 1
    return cnt

nfd_before = int(df["summary"].fillna("").apply(lambda s: unicodedata.normalize("NFC", str(s)) != str(s)).sum())
stuck_before = _count_case_stuck(df["summary"])
stuck_after_split = _count_case_stuck(df["summary"].fillna("").apply(split_case_boundaries))

print("=== MINH HỌA TÁCH TỪ DÍNH RANH GIỚI HOA/THƯỜNG ===")
demo_cases = [
    "bên tráiĐại Khư",
    "Nhĩ CănThể loại: Tiên hiệpGiới thiệu:",
    "1v1, HEEditor: Thảo",
    "WeChat, iPhone, YouTube (giữ nguyên whitelist)",
]
for raw_s in demo_cases:
    print(f"  {raw_s!r:48s} -> {split_case_boundaries(raw_s)!r}")

print(f"\nSố văn án chứa dấu tổ hợp NFD (chưa NFC)   : {nfd_before} -> 0 sau clean_text")
print(f"Số truyện có từ dính hoa/thường ngoài whitelist: {stuck_before} -> {stuck_after_split} sau split_case_boundaries()")

df["step1_clean"] = df["summary"].apply(clean_text)

print("\n=== ĐỐI CHIẾU BƯỚC 1 (LÀM SẠCH CƠ BẢN) ===")
for _, row in df.head(3).iterrows():
    print(f"[{row['id']}] {row['title']}")
    print("  Gốc       :", str(row["summary"])[:140])
    print("  Sau clean :", row["step1_clean"][:140])

=== MINH HỌA TÁCH TỪ DÍNH RANH GIỚI HOA/THƯỜNG ===
  'bên tráiĐại Khư'                                -> 'bên trái Đại Khư'
  'Nhĩ CănThể loại: Tiên hiệpGiới thiệu:'          -> 'Nhĩ Căn Thể loại: Tiên hiệp Giới thiệu:'
  '1v1, HEEditor: Thảo'                            -> '1v1, HE Editor: Thảo'
  'WeChat, iPhone, YouTube (giữ nguyên whitelist)' -> 'WeChat, iPhone, YouTube (giữ nguyên whitelist)'

Số văn án chứa dấu tổ hợp NFD (chưa NFC)   : 1036 -> 0 sau clean_text
Số truyện có từ dính hoa/thường ngoài whitelist: 6408 -> 0 sau split_case_boundaries()



=== ĐỐI CHIẾU BƯỚC 1 (LÀM SẠCH CƠ BẢN) ===
[ST_0001] Con Đường Bá Chủ
  Gốc       : Bạn đang đọc truyện Con Đường Bá Chủ full (đã hoàn thành) của tác giả Akay Hau. Truyện kể về Lạc Nam, một thanh niên xuất thân bình thường, 
  Sau clean : bạn đang đọc truyện con đường bá chủ full đã hoàn thành của tác giả akay hau truyện kể về lạc nam một thanh niên xuất thân bình thường bị gi
[ST_0002] Mục Thần Ký
  Gốc       : Edit: Góc trên bên tráiĐại Khư tổ huấn đã nói rõ, trời tối tốt nhất đừng ra ngoài.Đại Khư Tàn Lão Thôn, một đứa bé được những người già nhặt
  Sau clean : edit góc trên bên trái đại khư tổ huấn đã nói rõ trời tối tốt nhất đừng ra ngoài đại khư tàn lão thôn một đứa bé được những người già nhặt đ
[ST_0003] Vô Thượng Sát Thần
  Gốc       : Bạn đang đọc truyện Vô Thượng Sát Thần của tác giả Tà Tâm Vị Mẫn. Thế nào là thiên tài? Có thể lĩnh ngộ mọi chiến kỹ cùng công pháp, đây chí
  Sau clean : bạn đang đọc truyện vô thượng sát thần của tác giả tà tâm vị mẫn thế nào là thiên tài có 

In [3]:
# Bước 2: Chống rò rỉ nhãn bằng Regex (HEADER_GENRE_BLOCK_RE, LEADING_LABEL_PREFIX_RE, LEAKAGE_KEYWORDS)
df["summary_clean"] = df["step1_clean"].apply(remove_label_leakage)

print("=== KIỂM CHỨNG LOẠI RÒ RỈ NHÃN ===")
leak_samples = df[df["summary"].str.contains("Thể loại:", case=False, na=False)].head(3)
for _, row in leak_samples.iterrows():
    print(f"[{row['id']}] {row['title']}")
    print("  Trước leakage:", row["step1_clean"][:150])
    print("  Sau leakage  :", row["summary_clean"][:150])

nfd_after = int(df["summary_clean"].fillna("").apply(lambda s: unicodedata.normalize("NFC", str(s)) != str(s)).sum())
header_before = int(df["step1_clean"].fillna("").apply(lambda s: bool(HEADER_GENRE_BLOCK_RE.search(s))).sum())
header_after = int(df["summary_clean"].fillna("").apply(lambda s: bool(HEADER_GENRE_BLOCK_RE.search(s))).sum())

kw_word_re = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(unicodedata.normalize("NFC", k.lower())) for k in LEAKAGE_KEYWORDS if k.strip()) + r")(?!\w)",
    flags=re.IGNORECASE | re.UNICODE,
)
leak_kw_before = int(df["step1_clean"].fillna("").apply(lambda s: bool(kw_word_re.search(s))).sum())
leak_kw_after = int(df["summary_clean"].fillna("").apply(lambda s: bool(kw_word_re.search(s))).sum())

audit_df = pd.DataFrame(
    [
        {"chi_tieu_kiem_tra": "Văn án chứa ký tự tổ hợp NFD (summary -> summary_clean)", "truoc": nfd_before, "sau": nfd_after},
        {"chi_tieu_kiem_tra": "Truyện có từ dính hoa/thường ngoài CAMELCASE_WHITELIST", "truoc": stuck_before, "sau": stuck_after_split},
        {"chi_tieu_kiem_tra": "Văn án có cụm header 'Thể loại: ... Văn án/Giới thiệu:'", "truoc": header_before, "sau": header_after},
        {"chi_tieu_kiem_tra": "Văn án có từ khóa thể loại LEAKAGE_KEYWORDS (step1_clean -> summary_clean)", "truoc": leak_kw_before, "sau": leak_kw_after},
    ]
)
display(audit_df)

=== KIỂM CHỨNG LOẠI RÒ RỈ NHÃN ===
[ST_0014] Quang Âm Chi Ngoại
  Trước leakage: quang âm chi ngoại là tác phẩm mới nhất của nhĩ căn một trong những tác giả nổi tiếng trong thể loại tiên hiệp và huyền huyễn sau thành công của các t
  Sau leakage  : quang âm chi ngoại là tác phẩm mới nhất của nhĩ căn một trong những tác giả nổi tiếng trong và sau thành công của các tác phẩm đình đám như tiên nghịc
[ST_0015] Tuyệt Thế Dược Thần
  Trước leakage: dịch giả nhóm độc cô tử dạ thể loại dị thế tiên hiệp kiểm hiệp nội dung chính kể về con đường đan dược của nhân vật chính diệp viễn bá đạo từng bước t
  Sau leakage  : dịch giả nhóm độc cô tử dạ chính kể về con đường đan dược của nhân vật chính diệp viễn bá đạo từng bước tiến thân vượt qua mọi sóng gió vô cùng độc đá
[ST_0020] Tiên Võ Đế Tôn
  Trước leakage: tác giả lục giới tam đạo thể loại tiên hiệp kiếm hiệp dị giới huyền huyễn cổ đại chín ngàn năm trước chí tôn tiên đế suất lĩnh trăm vạn thần tướng nhố
  Sau leakage  : tác giả lục giới tam đạo

,chi_tieu_kiem_tra,truoc,sau
0,Văn án chứa ký tự tổ hợp NFD (summary -> summa...,1036,0
1,Truyện có từ dính hoa/thường ngoài CAMELCASE_W...,6408,0
2,Văn án có cụm header 'Thể loại: ... Văn án/Giớ...,2881,0
3,Văn án có từ khóa thể loại LEAKAGE_KEYWORDS (s...,9552,2


### Phân tích hiệu quả chuẩn hóa văn bản & Regex chống rò rỉ nhãn
- **Chuẩn hóa NFC & tách từ dính hoa/thường:** Xử lý triệt để **`1.036`** văn án chứa dấu tổ hợp `NFD` về chuẩn `NFC`, đồng thời đưa số truyện chứa chuỗi dính ranh giới hoa/thường (ngoài `CAMELCASE_WHITELIST`) từ **`6.408`** về **`0`** ngay trước bước `.lower()`.
- **Cắt header & loại từ khóa rò rỉ nhãn:** `HEADER_GENRE_BLOCK_RE` cắt sạch **`2.881 -> 0`** cụm khai báo mở đầu `"Thể loại: ... Văn án/Giới thiệu:"`, và giảm số văn án còn từ khóa trong `LEAKAGE_KEYWORDS` từ **`9.552` xuống `2`**.
- *(Chi tiết kiểm tra thêm sau bước tách từ `underthesea` được tổng hợp trong `bao_cao_ket_qua/02_tv2_tien_xu_ly_va_dac_trung.md`.)*

In [4]:
# Bước 3: Tách từ tiếng Việt & lọc Stopwords
stopwords = load_stopwords(STOPWORDS_PATH)
print(f"Đã nạp {len(stopwords)} từ dừng từ {STOPWORDS_PATH.name}")

df["tokens"] = df["summary_clean"].apply(lambda text: tokenize_vietnamese(text, stopwords=stopwords))

print("\n=== ĐỐI CHIẾU TÁCH TỪ & LỌC STOPWORDS ===")
for _, row in df.head(3).iterrows():
    print(f"[{row['id']}] {row['title']}")
    print("  summary_clean:", row["summary_clean"][:130])
    print("  tokens       :", row["tokens"][:130])

Đã nạp 120 từ dừng từ vietnamese_stopwords.txt



=== ĐỐI CHIẾU TÁCH TỪ & LỌC STOPWORDS ===
[ST_0001] Con Đường Bá Chủ
  summary_clean: bạn đang đọc truyện con đường bá chủ full đã hoàn thành của tác giả akay hau truyện kể về lạc nam một thanh niên xuất thân bình th
  tokens       : con đường bá_chủ akay hau_truyện kể lạc_nam thanh_niên xuất_thân bình_thường gia_tộc người_đời khinh_thường tư_chất kém tuy_nhiên 
[ST_0002] Mục Thần Ký
  summary_clean: edit góc trên bên trái đại khư tổ huấn đã nói rõ trời tối tốt nhất đừng ra ngoài đại khư tàn lão thôn một đứa bé được những người 
  tokens       : edit góc bên trái đại_khư tổ_huấn nói rõ trời tối tốt đừng ngoài đại_khư tàn_lão thôn đứa bé già nhặt ở bờ sống đặt tên tần_mục tâ
[ST_0003] Vô Thượng Sát Thần
  summary_clean: bạn đang đọc truyện vô thượng sát thần của tác giả tà tâm vị mẫn thế nào là thiên tài có thể lĩnh ngộ mọi chiến kỹ cùng công pháp 
  tokens       : vô thượng_sát_thần tà_tâm_vị_mẫn thế_nào thiên_tài có_thể lĩnh_ngộ chiến_kỹ công_pháp thiên_tài chân_chính mang thần bí_th

In [5]:
# Bước 4: Chia tập train/val/test (70/15/15) và xuất data/processed/stories_clean.csv
label_cols = [f"label_{g}" for g in GENRE_LABELS]
df["split"] = assign_splits(df, label_cols)

df_clean = df.drop(columns=["step1_clean"]).copy()
STORIES_CLEAN_CSV.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(STORIES_CLEAN_CSV, index=False, encoding="utf-8")

print(f"Đã xuất {len(df_clean)} mẫu sạch vào: {STORIES_CLEAN_CSV}")
print("Kiểm tra ô rỗng summary_clean / tokens / split:",
      int((df_clean["summary_clean"].fillna("").str.strip() == "").sum()),
      int((df_clean["tokens"].fillna("").str.strip() == "").sum()),
      int((df_clean["split"].fillna("").str.strip() == "").sum()))

split_stats = {}
for split_name in ["train", "val", "test"]:
    sub = df_clean[df_clean["split"] == split_name]
    split_stats[split_name] = {lbl: int(sub[f"label_{lbl}"].sum()) for lbl in GENRE_LABELS}
    split_stats[split_name]["TONG_MAU"] = len(sub)

display(pd.DataFrame(split_stats))
df_clean[["id", "title", "genres", "summary_clean", "tokens", "split"]].head()

Đã xuất 20852 mẫu sạch vào: C:\Users\ACER\Downloads\NLP_CUỐI KỲ\data\processed\stories_clean.csv
Kiểm tra ô rỗng summary_clean / tokens / split: 1 1 0


,train,val,test
ngon_tinh,6222,1721,1721
dam_my,2792,598,598
bach_hop,369,53,54
tong_tai,388,93,107
nu_phu,326,28,30
nu_cuong,2683,544,545
sung,3255,781,779
nguoc,997,382,335
hai_huoc,1521,326,326
tien_hiep,987,211,212


,id,title,genres,summary_clean,tokens,split
0,ST_0001,Con Đường Bá Chủ,tien_hiep|co_dai|huyen_huyen,bạn đang đọc truyện con đường bá chủ full đã h...,con đường bá_chủ akay hau_truyện kể lạc_nam th...,train
1,ST_0002,Mục Thần Ký,tien_hiep|huyen_huyen,edit góc trên bên trái đại khư tổ huấn đã nói ...,edit góc bên trái đại_khư tổ_huấn nói rõ trời ...,test
2,ST_0003,Vô Thượng Sát Thần,tien_hiep|xuyen_khong|huyen_huyen|di_gioi,bạn đang đọc truyện vô thượng sát thần của tác...,vô thượng_sát_thần tà_tâm_vị_mẫn thế_nào thiên...,train
3,ST_0004,Thần Đạo Đan Tôn,tien_hiep|huyen_huyen|trong_sinh,bạn đang đọc truyện thần đạo đan tôn của tác g...,truyện_thần đạo đan_tôn côn_đơn địa_phi lăng_h...,val
4,ST_0005,Tiên Nghịch,tien_hiep|kiem_hiep|huyen_huyen,bạn đang đọc truyện tiên nghịch của tác giả nh...,tiên_nghịch nhĩ căn vương_lâm thiếu_niên bình_...,train


In [6]:
# Bước 5: Fit ma trận TF-IDF duy nhất trên tập train (chạy khi đã cài scikit-learn)
train_df = df_clean[df_clean["split"] == "train"]
val_df = df_clean[df_clean["split"] == "val"]
test_df = df_clean[df_clean["split"] == "test"]

try:
    from src.features.bow_tfidf import fit_transform_tfidf

    vectorizer, x_train, x_val, x_test = fit_transform_tfidf(
        train_df["tokens"].fillna(""),
        val_df["tokens"].fillna(""),
        test_df["tokens"].fillna(""),
        save_path=TFIDF_VECTORIZER_PATH,
    )
    print("Kích thước ma trận TF-IDF:", x_train.shape, x_val.shape, x_test.shape)
    print("Đã lưu vectorizer tại:", TFIDF_VECTORIZER_PATH)
except ImportError:
    print("Lưu ý: Chưa cài scikit-learn nên tạm bỏ qua bước fit TF-IDF (cài bằng pip install -r requirements.txt).")

Kích thước ma trận TF-IDF: (14613, 10000) (3125, 10000) (3114, 10000)
Đã lưu vectorizer tại: C:\Users\ACER\Downloads\NLP_CUỐI KỲ\models\tfidf_vectorizer.pkl


## Tổng kết Giai đoạn 4 (Tiền xử lý & Trích xuất TF-IDF)
- Xuất thành công **`data/processed/stories_clean.csv`** (`20.852` dòng × `52` cột), chia phân tầng đa nhãn với `RANDOM_STATE = 42` thành `train = 14.613`, `val = 3.125`, `test = 3.114` (mọi nhãn trong cả 35 nhãn ở `val`/`test` đều có $\ge 10$ mẫu; chỉ có `1` mẫu `ST_12713` trong `train` bị rỗng `tokens` do toàn bộ văn án gốc nằm trong cụm header `"Thể loại:"`).
- Fit `TfidfVectorizer` (`ngram_range=(1, 2)`, `max_features=10000`, `sublinear_tf=True`) **chỉ trên tập `train`** (`(14613, 10000)`) và lưu tại `models/tfidf_vectorizer.pkl`.